# Finetuning Qwen3.5

> Originally, adapted from [Qwen3_5_(0_8B)_Vision.ipynb](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Qwen3_5_(0_8B)_Vision.ipynb#scrollTo=gGFzmplrEy9I)

![Qwen3.5](https://qianwen-res.oss-accelerate.aliyuncs.com/logo_qwen3.5.png)

Qwen3.5 features the following enhancement:

- **Unified Vision-Language Foundation**: Early fusion training on multimodal tokens achieves cross-generational parity with Qwen3 and outperforms Qwen3-VL models across reasoning, coding, agents, and visual understanding benchmarks.
- **Efficient Hybrid Architecture**: Gated Delta Networks combined with sparse Mixture-of-Experts deliver high-throughput inference with minimal latency and cost overhead.
- **Scalable RL Generalization**: Reinforcement learning scaled across million-agent environments with progressively complex task distributions for robust real-world adaptability.
- **Global Linguistic Coverage**: Expanded support to 201 languages and dialects, enabling inclusive, worldwide deployment with nuanced cultural and regional understanding.
- **Next-Generation Training Infrastructure**: Near-100% multimodal training efficiency compared to text-only training and asynchronous RL frameworks supporting massive-scale agent scaffolds and environment orchestration.

In [ ]:
import json
import warnings
from collections import defaultdict
from pathlib import Path

from PIL import Image
from tqdm.auto import tqdm
from unsloth import FastVisionModel

from staged_qwen3_5_scivqa.config import (
    BASE_DIR,
    COMPETITION_DATA_DIR,
    DATA_DIR,
    ENABLE_THINKING,
    LORA_CHECKPOINT_PARAGRAPH,
    MODEL_ID,
    PROMPT_PARAGRAPH,
    TEMPERATURE,
    TOKEN_BUDGETS,
    TOP_K,
    TOP_P,
    MIN_P,
)
from staged_qwen3_5_scivqa.conversation import convert_to_inference_conversation
from staged_qwen3_5_scivqa.context import get_paper_context

In [ ]:
MAX_NEW_TOKENS = TOKEN_BUDGETS["Paragraph"]["max_new_tokens"]
MAX_SEQUENCE_LENGTH = TOKEN_BUDGETS["Paragraph"]["max_sequence_length"]

LORA_CHECKPOINT = LORA_CHECKPOINT_PARAGRAPH

DATA_DIR.mkdir(exist_ok=True)
CATEGORY = "test"

CASE_DIR = COMPETITION_DATA_DIR / CATEGORY

STATE_FILE = DATA_DIR / f"submission_finetuning_{CATEGORY}__paragraph_state.json"

SUMMARY_CACHE_PATH = DATA_DIR / f"submission_finetuning_summary_{CATEGORY}_state.json"
EXTRACTION_CACHE_PATH = (
    DATA_DIR / f"submission_finetuning_extraction_{CATEGORY}_state.json"
)

<a name="Data"></a>
### 🧪 Data Preparation

To convert our Sci-ImageMiner VQA data into the format required by Qwen2-VL (specifically for use with Unsloth), we need to restructure the data into a "messages" format.

The Qwen/Unsloth format expects a list of conversations where the image and the text prompt are bundled together in the user role, and the ground truth is in the assistant role, as follows:

```python
[
    { "role": "user",
    "content": [{"type": "text",  "text": Q}, {"type": "image", "image": image} ]
    },
    { "role": "assistant",
    "content": [{"type": "text",  "text": A} ]
    },
]
```

In [ ]:
PROMPT = PROMPT_PARAGRAPH


def load_test_dataset(
    case_dir: Path, summary_cache: dict, extraction_cache: dict
) -> list[dict]:
    samples = []
    json_files = list(case_dir.rglob("*.json"))
    pbar = tqdm(json_files, desc="Converting Test to Qwen Format")

    for json_file in pbar:
        fullpath = str(json_file)
        if (
            "content.json" in json_file.name
            or "images" not in fullpath
            or ".vscode" in fullpath
        ):
            continue

        pbar.set_description(f"Processing {json_file.name}")

        with open(json_file) as f:
            data = json.load(f)

        sample_id = data["sample_id"]
        img_path = json_file.with_suffix(".jpg")
        if not img_path.exists():
            continue

        # 2. Open source image once per JSON file
        full_img = Image.open(img_path.absolute())
        context = get_paper_context(json_file)
        bboxes = data.get("bbox", {})

        # 3. Iterate through subfigures (a, b, etc.)
        for sub_fig, q_list in data.get("vqa", {}).items():
            # 4. Check for bounding box and crop
            if sub_fig not in bboxes:
                warnings.warn(f"Subfigure {sub_fig} missing bbox in {json_file.name}")
                continue

            box = bboxes[sub_fig]
            left = box["x"]
            top = box["y"]
            right = left + box["width"]
            bottom = top + box["height"]

            sub_image = full_img.crop((left, top, right, bottom))

            summary = summary_cache.get(sample_id, {}).get(sub_fig)
            table = extraction_cache.get(sample_id, {}).get(sub_fig)

            for q_obj in q_list:
                question_text = q_obj.get("question") or q_obj.get("questions")
                question_type = q_obj.get("question_type", "")
                answer_type = q_obj.get("answer_type", "")

                if answer_type != "Paragraph":
                    continue

                human_prompt = PROMPT.format(
                    question=question_text,
                    question_type=question_type,
                    context=context,
                    summary=summary
                    if summary is not None
                    else "N/A",  # Aligned with training code fallback
                    table=table
                    if table is not None
                    else "N/A",  # Aligned with training code fallback
                )

                # 5. Format for inference (User role only)
                sample = convert_to_inference_conversation(
                    human_prompt,
                    sub_image,
                    sample_id=sample_id,
                    sub_fig=sub_fig,
                    question_type=question_type,
                    answer_type=answer_type,
                    question=question_text,
                )

                samples.append(sample)

    return samples

Let's convert the dataset into the "correct" format for finetuning:

In [ ]:
summary_cache = None
with open(SUMMARY_CACHE_PATH) as f:
    summary_cache = json.load(f)

extraction_cache = None
with open(EXTRACTION_CACHE_PATH) as f:
    extraction_cache = json.load(f)

dataset = load_test_dataset(CASE_DIR, summary_cache, extraction_cache)

We look at how the conversations are structured for the first example:

In [ ]:
dataset[0]["messages"][0]["content"][1]["image"]

In [ ]:
dataset[0]

<a name="Submission"></a>
### 📜 Update the submission

Let's now update our submission! First, we must load the LoRA adapters we saved for inference!

In [ ]:
paragraph_state = defaultdict(lambda: defaultdict(dict))

if STATE_FILE.exists():
    with open(STATE_FILE) as f:
        saved_paragraph_state = json.load(f)

    # Reconstruct completed_tasks and paragraph_state
    for s_id, sub_figs in saved_paragraph_state.items():
        for s_fig, questions in sub_figs.items():
            # Handle the new dictionary format
            for q_text, ans_text in questions.items():
                task_id = f"{s_id}::{s_fig}::{q_text}"
                paragraph_state[s_id][s_fig][q_text] = ans_text

    print(f"Loaded tracking state from {STATE_FILE}.")

In [ ]:
model, tokenizer = FastVisionModel.from_pretrained(
    model_name=LORA_CHECKPOINT,
    load_in_4bit=True,  # Set to False for 16bit LoRA
    max_seq_length=MAX_SEQUENCE_LENGTH,  # Must match the max_length used during training
)
FastVisionModel.for_inference(model)  # Enable for inference!

In [ ]:
pbar = tqdm(dataset, desc=f"Processing {CATEGORY} split", position=1, leave=False)
processed_cnt = 0
already_processed_cnt = 0

for data_item in pbar:
    meta = data_item["meta"]
    sample_id = meta["sample_id"]
    sub_fig = meta["sub_fig"]
    question_text = meta["question"]
    question_type = meta["question_type"]

    task_id = f"{sample_id}::{sub_fig}::{question_text}"

    # Check if already processed in a previous interrupted run
    if paragraph_state.get(sample_id, {}).get(sub_fig, {}).get(question_text):
        already_processed_cnt += 1
        pbar.set_postfix(processed=processed_cnt, already=already_processed_cnt)
        continue

    pbar.set_description(f"Generating: {sample_id} | Sub: {sub_fig}")

    # Extract messages and image directly from the pre-built dataset item
    messages = data_item["messages"]
    image = messages[0]["content"][1]["image"]

    # Tokenize and format
    input_text = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, enable_thinking=ENABLE_THINKING
    )
    inputs = tokenizer(
        image,
        input_text,
        add_special_tokens=False,
        return_tensors="pt",
    ).to("cuda")

    output_ids = model.generate(
        **inputs,
        max_new_tokens=MAX_NEW_TOKENS,
        use_cache=True,
        temperature=TEMPERATURE,
        min_p=MIN_P,
        top_p=TOP_P,
        top_k=TOP_K,
    )

    # Decode only the newly generated tokens
    generated = tokenizer.decode(
        output_ids[0][inputs["input_ids"].shape[-1] :],
        skip_special_tokens=True,
    ).strip()

    # 1. Update the Paragraph-Specific Tracking State
    paragraph_state[sample_id][sub_fig][question_text] = generated

    processed_cnt += 1
    pbar.set_postfix(processed=processed_cnt, already=already_processed_cnt)

    # 3. Persist State Every X Iterations (e.g., 5)
    if processed_cnt % 5 == 0:
        with open(STATE_FILE, "w") as f:
            json.dump(paragraph_state, f, indent=4)

with open(STATE_FILE, "w") as f:
    json.dump(paragraph_state, f, indent=4)

print(
    f"Pipeline execution complete! State fully synced.
"
    f"Summary -> Processed (Overwritten): {processed_cnt} | Already Completed: {already_processed_cnt}"
)